# The LoRA Parametrization (Research POV)

**Thesis.** Unit 1 proved low rank is a *measured, exactly-quantified* property: `W_gate` needs 394/576 directions for 90% energy, and dropping the rest costs a predictable ~31% error. Unit 2 builds the **adaptation machinery** on top: the `ΔW = (α/r)·B·A` parametrization, its parameter budget, and the merge trick that makes a low-rank adapter *free at inference*. By the end you can compute, for any weight shape, exactly what a LoRA adapter costs — in parameters, bytes, and FLOPs.

**Repo position.** Units 1 and 2 are the foundation: *low rank is real* (Unit 1) and *low-rank adaptation is cheap and mergeable* (Unit 2). Unit 3 then tests the hypothesis with real gradient descent.

### Setup — load the model, extract the matrices

Same self-contained load as Unit 1: SmolLM2-135M, cast to float32, extract the five matrices and the token embeddings `X`. Units 2–4 reuse these exact shapes.

In [1]:
# --- Setup: load SmolLM2-135M, extract the matrices (float32, CPU) ---
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
import numpy as np

model = AutoModelForCausalLM.from_pretrained(
    "HuggingFaceTB/SmolLM2-135M",
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
)
model.eval()

tok = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-135M")

sd = model.state_dict()
W_gate = sd["model.layers.0.mlp.gate_proj.weight"].float().numpy()   # (1536, 576)
W_up   = sd["model.layers.0.mlp.up_proj.weight"].float().numpy()     # (1536, 576)
W_down = sd["model.layers.0.mlp.down_proj.weight"].float().numpy()   # (576, 1536)
W_Q    = sd["model.layers.0.self_attn.q_proj.weight"].float().numpy()  # (576, 576)
W_E    = sd["model.embed_tokens.weight"].float().numpy()             # (49152, 576)

ids = tok("The quick brown fox jumps over the lazy dog", return_tensors="pt").input_ids
with torch.no_grad():
    X = model.model.embed_tokens(ids)[0].float().numpy()              # (6, 576)

names = {"W_gate": W_gate, "W_up": W_up, "W_down": W_down, "W_Q": W_Q, "W_E": W_E}
for n, W in names.items():
    print(f"{n:8s} {str(W.shape):14s} dtype={W.dtype}")

del sd
print("X", X.shape, "| model loaded OK")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

W_gate   (1536, 576)    dtype=float32
W_up     (1536, 576)    dtype=float32
W_down   (576, 1536)    dtype=float32
W_Q      (576, 576)     dtype=float32
W_E      (49152, 576)   dtype=float32
X (9, 576) | model loaded OK


## Concept — The LoRA parametrization

Full fine-tuning updates *every* element of a weight:

$$W_{\text{new}} = W_0 + \Delta W, \qquad \Delta W \in \mathbb{R}^{d \times k} \;(d\cdot k \text{ new params})$$

**LoRA's bet:** the update fine-tuning actually needs is low-rank, so factor it:

$$\Delta W = \tfrac{\alpha}{r}\, B\, A, \qquad B \in \mathbb{R}^{d \times r},\; A \in \mathbb{R}^{r \times k}, \; r \ll \min(d,k)$$

and apply the forward pass as:

$$W_{\text{eff}}\,x = W_0\,x + \tfrac{\alpha}{r}\, B\,(A\,x)$$

- `r` — the rank budget (commonly 8). `α` — a scalar that scales the update; the `α/r` ratio behaves like a learning-rate scale.
- **Initialization:** `A` ~ small Gaussian, `B = 0`. At step zero `ΔW = 0`, so the adapter starts as a **no-op** and only grows the model along the directions the task needs.
- **Why the bet is plausible:** Unit 1's spectra. If fine-tuning on a narrow task moves `W` mostly along a few directions, an `r`-dimensional delta can carry most of that movement — at ~2% of the parameters.

**The measured question:** how cheap is `ΔW = (α/r)B·A` on SmolLM2's real shapes, and how small is `ΔW` relative to `W`?

In [19]:
# LoRA params  = d·r + r·k = r·(d + k) | 576×8 + 8×576 = 9,216 cells
# full params  = d·k | 576×576 = 331,776 cells (full)
# pct = 100 × lo / full = 100 × 16,896 / 884,736 = 1.91%
# KB = 2 × 9216/1024 = 18 KB

print(f"LoRA : adapter weight count\nfull : base layer params\npct : parameter compression ratio \nr : lowest rank\n")
for name, W in names.items():
    d, k = W.shape
    full = d * k
    for r in [2, 4, 8, 16, 64]:     
        lo = d * r + r * k          # B (d x r) + A (r x k)
        pct = 100 * lo / full
        print(f"{name:7s} {str(W.shape):14s} r={r:2d}  "
              f"LoRA={lo:9d}  full={full:10d}  "
              f"pct={pct:6.2f}%  bf16={2*lo/1024:7.1f}KB")
    print("\n")
              

LoRA : adapter weight count
full : base layer params
pct : parameter compression ratio 
r : lowest rank

W_gate  (1536, 576)    r= 2  LoRA=     4224  full=    884736  pct=  0.48%  bf16=    8.2KB
W_gate  (1536, 576)    r= 4  LoRA=     8448  full=    884736  pct=  0.95%  bf16=   16.5KB
W_gate  (1536, 576)    r= 8  LoRA=    16896  full=    884736  pct=  1.91%  bf16=   33.0KB
W_gate  (1536, 576)    r=16  LoRA=    33792  full=    884736  pct=  3.82%  bf16=   66.0KB
W_gate  (1536, 576)    r=64  LoRA=   135168  full=    884736  pct= 15.28%  bf16=  264.0KB


W_up    (1536, 576)    r= 2  LoRA=     4224  full=    884736  pct=  0.48%  bf16=    8.2KB
W_up    (1536, 576)    r= 4  LoRA=     8448  full=    884736  pct=  0.95%  bf16=   16.5KB
W_up    (1536, 576)    r= 8  LoRA=    16896  full=    884736  pct=  1.91%  bf16=   33.0KB
W_up    (1536, 576)    r=16  LoRA=    33792  full=    884736  pct=  3.82%  bf16=   66.0KB
W_up    (1536, 576)    r=64  LoRA=   135168  full=    884736  pct= 15.28%  bf16=  2

## Concept — The parameter budget, and the size of the delta

Full update stores `d·k` numbers. LoRA stores `d·r + r·k`:

$$\underbrace{d\cdot r}_{B} + \underbrace{r\cdot k}_{A} \;\ll\; d\cdot k$$

On real SmolLM2 shapes at `r = 8`:

- `W_Q` 576×576: 576·8 + 8·576 = 9,216 vs 331,776 → **~2.8%**
- `W_gate` 1536×576: 1536·8 + 8·576 = 16,896 vs 884,736 → **~1.9%**

Stored in bf16 (2 bytes), a `W_gate` adapter is ~33 KB. That is the memory and storage cost of adapting — and it is the whole point: adapt many tasks, keep the base frozen and shared.

**But `ΔW` must also be small in *energy*.** Unit 1's spectrum says rank-8 holds almost none of `W_gate`'s energy (k=8 → ~96% reconstruction error). That is exactly right: LoRA is a **modifier, not a copy**. measure the perturbation size directly, and — because `ΔW = B·A` is a product of two matrices of rank ≤ `r` — it is **confined to `r` directions by construction**.

**Two properties to verify:** (1) `rank(ΔW) ≤ r` (SVD will show it), (2) `‖ΔW‖_F / ‖W‖_F` is small (the delta is a nudge, not a rewrite) | result : delta's energy is 0.61%

In [15]:
# build dW = (alpha/r) B A; check rank and size
W = W_gate
d, k = W.shape
r, alpha = 8, 16
scale = alpha / r

A = np.random.default_rng(0).normal(0, 0.02, size=(r,k))
B = np.zeros((d, r))        # B = 0 -> dW = 0 at init
dW0 = scale * (B @ A)
print("init delta norm:", np.linalg.norm(dW0))     # expect 0.0 (no-op start)

B = np.random.default_rng(1).normal(0, 0.01, size=(d,r))    # "trained" B
dW = scale * (B @ A)

s = np.linalg.svd(dW, compute_uv=False)
print("nonzero singular values:", int(np.sum(s > 1e-6 * s[0])), "vs r =", r)
print("||dW||_F / ||W||_F : (delta's energy) =", np.linalg.norm(dW) / np.linalg.norm(W))

init delta norm: 0.0
nonzero singular values: 8 vs r = 8
||dW||_F / ||W||_F : (delta's energy) = 0.006131619574292975


## Concept — The merge trick

Once trained, `B` and `A` are fixed. Two ways to run the adapter:

**Merged (deployment default):** precompute `W_eff = W_0 + (α/r)·B·A` once, then serve `W_eff` as an ordinary weight — **zero extra FLOPs per token**.

**Unmerged (multi-adapter routing):** keep `W_0` shared and apply `W_0 x + (α/r)·B(A x)` per adapter. Extra cost: two small matmuls per adapter, but switching tasks is instant and the base weight is never duplicated.

Both are mathematically identical — verifying it numerically on a real matrix and a real token.

In [14]:
# merge trick
W_eff = W_gate +  scale * (B @ A)

x0 = X[0]
merged  = W_eff @ x0                               # 1 - matmul
unmerged = W_gate @ x0 + scale * (B @ (A @ x0))    # 2 - 2 small matmuls

err = np.max(np.abs(merged - unmerged))
print("max |merged - unmerged|:", err)    # expect ~1e-6 (float noise)

max |merged - unmerged|: 2.1886080503463745e-07


## Measured findings (SmolLM2-135M, not assumed)

| # | Claim | Predicted | Measured | Verdict |
|---|---|---|---|---|
| 1 | LoRA params are a small fraction of W | ~2-3% at r=8 | CB 2.1 | ✅ |
| 2 | `ΔW` is exactly rank ≤ r | ≤ 8 by construction | CB 2.2 | ✅ |
| 3 | `ΔW` is a small perturbation of W | ‖ΔW‖/‖W‖ ~ 1e-2..1e-3 | CB 2.2 | ✅ |
| 4 | Merged == unmerged | max err ~ 1e-6 | CB 2.3 | ✅ |

Fill the Measured and Verdict columns from your actual runs before finishing.

## Conclusion

| Claim | Predicted | Measured | Verdict |
|---|---|---|---|
| Adapting costs a few percent of params | ~2% at r=8 | CB 2.1 | ✅ |
| The delta is low-rank and small | rank ≤ r, ‖ΔW‖ << ‖W‖ | CB 2.2 | ✅ |
| Merged and unmerged are identical | err ~ 1e-6 | CB 2.3 | ✅ |

**One-line takeaway (fill in your measured numbers, then rewrite).** LoRA is Unit 1's spectrum turned into a budget: a rank-8 delta `(α/r)·B·A` costs ~2% of the parameters, is confined to 8 directions, stays small relative to `W`, and merges into `W` for zero inference overhead.

**Next — Unit 3:** the hypothesis itself — train a real LoRA (r=8) on one layer with gradient descent and SVD the *learned* `ΔW`. Does the optimizer actually find a low-rank delta?

## Industrial note — usage, scenarios

- **The 2% number is the pitch.** “Adapt with ~2% of the parameters” is a budget figure — and you can now compute it for any shape with one formula.
- **Merge vs no-merge is an engineering decision.** Single-task serving merges (zero overhead); multi-task routing keeps deltas separate. Both are verified here; choose with data.
- **Storage math generalizes.** Adapter size in bytes is `(d·r + r·k)·bytes` — trivial at any scale, which is why LoRA adapters are the default interchange format for fine-tuned models.
